# 03 · Euler Angles and Gimbal Lock

### Recap & why now
Roll, pitch and yaw are how humans talk about orientation, and for good reason —
"fifteen degrees of bank" means something, while a list of four numbers does not.

They are a poor choice for *simulating*, and the reason is not aesthetic. At one
particular attitude the three angles stop describing three independent things, and a
simulator that integrates them there produces nonsense. This notebook demonstrates
that rather than asserting it.

### Learning objectives
1. Define roll, pitch and yaw with an explicit **order convention**.
2. Convert between Euler angles and a rotation matrix, both ways.
3. Demonstrate **gimbal lock** as two different angle sets giving the same matrix.
4. Show the Euler-rate map blowing up near the singularity.
5. State when Euler angles are still the right tool.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

## 1 · Three angles, one order

$$R(\phi, \theta, \psi) = R_z(\psi)\, R_y(\theta)\, R_x(\phi)$$

Read right to left: **roll first, then pitch, then yaw** — the aerospace "ZYX"
convention. Notebook 02 showed why the order has to be stated: a different order is a
different rotation.

$\phi$ is roll about the nose, $\theta$ is pitch about the left wing, $\psi$ is yaw
about the thrust axis.

In [ ]:
def Rx(a): c_, s_ = np.cos(a), np.sin(a); return np.array([[1,0,0],[0,c_,-s_],[0,s_,c_]])
def Ry(a): c_, s_ = np.cos(a), np.sin(a); return np.array([[c_,0,s_],[0,1,0],[-s_,0,c_]])
def Rz(a): c_, s_ = np.cos(a), np.sin(a); return np.array([[c_,-s_,0],[s_,c_,0],[0,0,1]])

def euler_to_rotmat(roll, pitch, yaw):
    """ZYX: apply roll first, then pitch, then yaw."""
    return Rz(yaw) @ Ry(pitch) @ Rx(roll)          # Right-to-left is the order of application.

roll, pitch, yaw = 0.35, -0.20, 1.10
print("Euler matrix and quaternion matrix agree to %.2e ✔" %
      np.abs(euler_to_rotmat(roll, pitch, yaw) - quat_to_rotmat(euler_to_quat(roll, pitch, yaw))).max())
print("round trip back through quat_to_euler: %.2e" %
      np.abs(quat_to_euler(euler_to_quat(roll, pitch, yaw)) - np.array([roll, pitch, yaw])).max())

print("\n  roll  pitch   yaw     thrust axis in the world")
for r, p_, y_ in [(0, 0, 0), (30, 0, 0), (0, 30, 0), (0, 0, 30), (20, 20, 45)]:
    R = euler_to_rotmat(*np.deg2rad([r, p_, y_]))
    print("  %4d %6d %5d     %s" % (r, p_, y_, np.round(R[:, 2], 3)))

## 2 · The hole

At $\theta = \pm 90°$ — the drone pointing straight up or straight down — the roll axis
and the yaw axis line up. Rolling and yawing then do the *same thing*, and one degree
of freedom silently vanishes.

It has a name, **gimbal lock**, and the demonstration is an equality: two genuinely
different angle triples produce the identical rotation matrix.

In [ ]:
A = euler_to_rotmat(np.deg2rad(0),   np.deg2rad(90), np.deg2rad(30))   # Yaw 30, no roll.
B = euler_to_rotmat(np.deg2rad(-30), np.deg2rad(90), np.deg2rad(0))    # Roll -30, no yaw.

print("(roll, pitch, yaw) = (  0, 90,  30)")
print("(roll, pitch, yaw) = (-30, 90,   0)")
print("largest difference between the two matrices: %.2e" % np.abs(A - B).max())
print("-> they are the SAME orientation. Roll and yaw have collapsed into one knob.\n")

print("  pitch    do roll and yaw still do different things?")
for deg in (0, 45, 80, 89, 89.9):
    th = np.deg2rad(deg)
    r_axis = euler_to_rotmat(0, th, 0) @ np.array([1.0, 0, 0])       # Where a roll would act.
    y_axis = np.array([0.0, 0, 1.0])                                 # Where a yaw acts.
    print("  %5.1f° %20.4f  (1.0 = identical, 0.0 = independent)" %
          (deg, abs(float(r_axis @ y_axis))))
print("\nThe two axes become parallel as the pitch approaches 90°, which is the collapse")
print("happening continuously rather than suddenly. Nothing warns you on the way in.")

## 3 · Why a simulator cannot survive it

The singularity is not merely a labelling nuisance. To integrate Euler angles you need
the map from **body rates** to **Euler rates**, and that map contains $1/\cos\theta$:

$$\begin{bmatrix} \dot\phi \\ \dot\theta \\ \dot\psi \end{bmatrix}
= \begin{bmatrix}
1 & \sin\phi\tan\theta & \cos\phi\tan\theta \\
0 & \cos\phi & -\sin\phi \\
0 & \sin\phi/\cos\theta & \cos\phi/\cos\theta
\end{bmatrix} \begin{bmatrix} p \\ q \\ r \end{bmatrix}$$

Near $\theta = 90°$ those terms explode. A perfectly gentle drone would need Euler
rates in the hundreds, and a fixed-step integrator asked to follow them returns
garbage — in one step, with no warning.

In [ ]:
def euler_rate_matrix(roll, pitch):
    """Body rates -> Euler rates. The 1/cos(pitch) terms are the problem."""
    sr, cr = np.sin(roll), np.cos(roll)
    tp, cp = np.tan(pitch), np.cos(pitch)
    return np.array([[1, sr*tp, cr*tp],
                     [0, cr, -sr],
                     [0, sr/cp, cr/cp]])           # These two entries are what blow up.

body_rate = np.array([1.0, 1.0, 1.0])              # A modest 1 rad/s about each body axis.
print("  pitch      largest Euler rate demanded")
for deg in (0, 60, 85, 89, 89.9, 89.99):
    W = euler_rate_matrix(0.2, np.deg2rad(deg))
    print("  %6.2f° %22.1f rad/s" % (deg, np.abs(W @ body_rate).max()))

fig, ax = plt.subplots(figsize=(7.4, 3.0))
degs = np.linspace(0, 89.5, 400)
worst = [np.abs(euler_rate_matrix(0.2, np.deg2rad(d_)) @ body_rate).max() for d_ in degs]
ax.plot(degs, worst, color="C3", lw=2.2)
ax.set_yscale("log"); ax.set_xlabel("pitch [deg]"); ax.set_ylabel("required Euler rate [rad/s]")
ax.set_title("A gentle drone, an impossible demand")
plt.show()
print("At 89.99° a 1 rad/s rotation needs Euler rates near %.0f. No fixed time step survives that." %
      max(worst))

## 4 · When Euler angles are still right

None of this makes them useless — it makes them a **display** format rather than a
state format.

Use them for telemetry, for plots, for talking to a human, and for setting an initial
attitude. Do not integrate them, and do not store the simulator's orientation in them.
Everything in this project follows that rule: the state carries a quaternion, and
`quat_to_euler` appears only in print statements and axis labels.

In [ ]:
print("  what you want to do                     use")
for job, tool in [("tell a pilot the bank angle       ", "Euler"),
                  ("plot attitude over time           ", "Euler"),
                  ("set a starting pose               ", "Euler"),
                  ("integrate a gyroscope             ", "quaternion"),
                  ("store the simulator's state       ", "quaternion"),
                  ("compute an attitude error         ", "quaternion")]:
    print("  %s %s" % (job, tool))

q = euler_to_quat(np.deg2rad(12), np.deg2rad(-7), np.deg2rad(80))
print("\nexample: the state carries %s" % np.round(q, 4))
print("         the telemetry shows roll %.1f°, pitch %.1f°, yaw %.1f°" %
      tuple(np.degrees(quat_to_euler(q))))
print("\nSame orientation, two representations, each doing the job it is good at.")

## 🧪 Try it yourself

**E1.** Gimbal lock happens at pitch $\pm 90°$ for the ZYX convention. Does choosing a
different order remove the problem, or move it?

**E2.** Integrate Euler angles through a manoeuvre that passes close to vertical and
compare against a quaternion doing the same job.

In [ ]:
# --- Solution E1 ---
print("E1: it moves it. Every three-angle parameterisation of 3-D rotation has a singularity")
print("    somewhere — this is a topological fact, not a failure of imagination: you cannot cover")
print("    the space of rotations smoothly with three numbers. ZYX puts the hole at pitch ±90°,")
print("    which is convenient for aircraft that rarely point straight up, and terrible for")
print("    aerobatic drones that do. Four numbers with one constraint — a quaternion — has no hole.")

# --- Solution E2 ---
w_body = np.array([0.0, 1.6, 0.0])                 # A steady pitch rate: straight toward vertical.
dt, n = 0.002, 1400
euler = np.array([0.05, 0.0, 0.0]); q = euler_to_quat(*euler)
euler_hist, quat_hist = [], []
for k in range(n):
    W = euler_rate_matrix(euler[0], euler[1])
    euler = euler + dt*(W @ w_body)                # Integrating the ANGLES: the fragile way.
    q = quat_normalize(q + dt*0.5*quat_multiply(q, np.array([0.0, *w_body])))   # The robust way.
    euler_hist.append(euler.copy()); quat_hist.append(quat_to_euler(q))
euler_hist = np.array(euler_hist); quat_hist = np.array(quat_hist)

fig, ax = plt.subplots(figsize=(7.4, 3.0))
t = np.arange(n)*dt
ax.plot(t, np.degrees(euler_hist[:, 0]), color="C3", lw=1.8, label="roll, Euler-integrated")
ax.plot(t, np.degrees(quat_hist[:, 0]), color="C0", lw=1.8, label="roll, quaternion-integrated")
ax.set_ylim(-200, 200); ax.set_xlabel("time [s]"); ax.set_ylabel("roll [deg]"); ax.legend(fontsize=9)
ax.set_title("Passing through vertical")
plt.show()

k90 = np.argmax(np.degrees(quat_hist[:, 1]) > 89.0)
print("E2: the drone reaches 89° of pitch at t = %.2f s." % (k90*dt))
print("    At that moment the Euler integration reports a roll of %.1f° while the quaternion" %
      np.degrees(euler_hist[k90, 0]))
print("    version reports %.1f°. The drone is doing one thing; one of the two descriptions has" %
      np.degrees(quat_hist[k90, 0]))
print("    stopped tracking it. Nothing errored, nothing warned — the numbers simply became wrong.")

## 🚁 Mini-project: flying into the singularity

Animate the manoeuvre from E2, showing the drone and both attitude read-outs. Watch
them agree, then diverge as the pitch approaches vertical.

In [ ]:
step = 12
fig = plt.figure(figsize=(9.8, 4.4))

def frame(j):
    fig.clf()
    k = step*j
    ax = fig.add_subplot(121, projection="3d")
    draw_quad(ax, [0, 0, 0], euler_to_quat(*quat_hist[k]), scale=2.2)
    set_3d(ax, (-1.2, 1.2), (-1.2, 1.2), (-1.0, 1.2))
    ax.set_title("pitch %5.1f°" % np.degrees(quat_hist[k, 1]), fontsize=10)
    ax.view_init(elev=18, azim=-62)
    ax2 = fig.add_subplot(122)
    ax2.plot(t[:k+1], np.degrees(euler_hist[:k+1, 0]), color="C3", lw=1.6, label="Euler roll")
    ax2.plot(t[:k+1], np.degrees(quat_hist[:k+1, 0]), color="C0", lw=1.6, label="quaternion roll")
    ax2.set_xlim(0, t[-1]); ax2.set_ylim(-200, 200); ax2.legend(fontsize=8)
    ax2.set_xlabel("time [s]"); ax2.set_ylabel("roll [deg]")
    return []

anim = animation.FuncAnimation(fig, frame, frames=n//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Gimbal lock is named after the mechanical failure it
> mirrors: three nested gimbal rings on an inertial platform can align and lose an axis,
> which happened on Apollo 11 and is why the spacecraft carried a "gimbal lock" warning
> light. Modern systems avoid the mechanism and the mathematics alike — orientation is
> carried as a quaternion or a matrix, and Euler angles appear only on the display.

**Where next.** Notebook 04 introduces the representation without a hole: four numbers,
one constraint, and no singularity anywhere.